# Silver Validation Notebook (Workshop 2)

This notebook validates the Silver layer outputs generated by the Airflow DAG.
It reads the three Parquet datasets from local disk and runs lightweight quality checks.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

In [ ]:
# Local paths for this repository layout
silver_dir = Path('../datalake_silver').resolve()
silver_dir

In [ ]:
def latest_parquet(prefix: str, base_dir: Path) -> Path:
    files = sorted(base_dir.glob(f'{prefix}_*.parquet'))
    if not files:
        raise FileNotFoundError(f'No files found for prefix: {prefix}')
    return files[-1]

titles_path = latest_parquet('silver_titles', silver_dir)
rt_reviews_path = latest_parquet('silver_rt_reviews', silver_dir)
tmdb_reviews_path = latest_parquet('silver_tmdb_reviews', silver_dir)

titles_path, rt_reviews_path, tmdb_reviews_path

In [ ]:
import pandas as pd
from pathlib import Path

silver_dir = Path(r"C:\Users\Julian Garcia\Documents\Uni\10\P.A.D\Streaming_Data_Analysis\datalake_silver")

for f in sorted(silver_dir.glob("*.parquet")):
    print(f"\nTrying: {f.name}")
    try:
        df = pd.read_parquet(f, engine="pyarrow")
        print("OK:", df.shape)
    except Exception as e:
        print("FAIL:", type(e).__name__, "-", e)

In [ ]:
df_titles = pd.read_parquet(titles_path)
df_rt_reviews = pd.read_parquet(rt_reviews_path)
df_tmdb_reviews = pd.read_parquet(tmdb_reviews_path)

print('Loaded datasets:')
print(f'silver_titles: {df_titles.shape}')
print(f'silver_rt_reviews: {df_rt_reviews.shape}')
print(f'silver_tmdb_reviews: {df_tmdb_reviews.shape}')

## 1) Shape and Columns

In [ ]:
datasets = {
    'silver_titles': df_titles,
    'silver_rt_reviews': df_rt_reviews,
    'silver_tmdb_reviews': df_tmdb_reviews,
}

for name, df in datasets.items():
    print('=' * 80)
    print(name)
    print('shape:', df.shape)
    print('columns:', list(df.columns))

## 2) Null Analysis

In [ ]:
def null_summary(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        'null_count': df.isna().sum(),
        'null_pct': (df.isna().mean() * 100).round(2),
    }).sort_values('null_pct', ascending=False)
    return out

null_titles = null_summary(df_titles)
null_rt = null_summary(df_rt_reviews)
null_tmdb = null_summary(df_tmdb_reviews)

null_titles.head(15), null_rt.head(15), null_tmdb.head(15)

## 3) Duplicate Checks (Business Keys)

In [ ]:
dup_titles = df_titles.duplicated(subset=['source', 'source_title', 'url', 'source_id']).sum()
dup_rt = df_rt_reviews.duplicated(subset=['canonical_title', 'review_type', 'reviewer_name', 'review_date', 'review_text']).sum()
dup_tmdb = df_tmdb_reviews.duplicated(subset=['movie_id', 'review_id']).sum()

pd.DataFrame({
    'dataset': ['silver_titles', 'silver_rt_reviews', 'silver_tmdb_reviews'],
    'duplicate_rows': [dup_titles, dup_rt, dup_tmdb],
})

## 4) Source Distribution Checks

In [ ]:
titles_source_dist = df_titles['source'].value_counts(dropna=False).to_frame('rows')
rt_review_type_dist = df_rt_reviews['review_type'].value_counts(dropna=False).to_frame('rows')
rt_source_type_dist = df_rt_reviews['bronze_source_type'].value_counts(dropna=False).to_frame('rows')
tmdb_source_dist = df_tmdb_reviews['source'].value_counts(dropna=False).to_frame('rows')

titles_source_dist, rt_review_type_dist, rt_source_type_dist, tmdb_source_dist

## 5) Validate review_text vs review_text_clean

In [ ]:
def text_clean_checks(df: pd.DataFrame, text_col: str = 'review_text', clean_col: str = 'review_text_clean') -> pd.DataFrame:
    total = len(df)
    both_present = df[text_col].notna() & df[clean_col].notna()
    changed = both_present & (df[text_col] != df[clean_col])

    return pd.DataFrame({
        'metric': [
            'total_rows',
            'raw_text_not_null',
            'clean_text_not_null',
            'both_present',
            'changed_after_cleaning',
        ],
        'value': [
            total,
            int(df[text_col].notna().sum()),
            int(df[clean_col].notna().sum()),
            int(both_present.sum()),
            int(changed.sum()),
        ],
    })

rt_text_checks = text_clean_checks(df_rt_reviews)
tmdb_text_checks = text_clean_checks(df_tmdb_reviews)

rt_text_checks, tmdb_text_checks

In [ ]:
# Show sample rows where cleaning changed content
rt_changed = df_rt_reviews[(df_rt_reviews['review_text'].notna()) & (df_rt_reviews['review_text_clean'].notna()) & (df_rt_reviews['review_text'] != df_rt_reviews['review_text_clean'])]
tmdb_changed = df_tmdb_reviews[(df_tmdb_reviews['review_text'].notna()) & (df_tmdb_reviews['review_text_clean'].notna()) & (df_tmdb_reviews['review_text'] != df_tmdb_reviews['review_text_clean'])]

print('RT changed samples:')
display(rt_changed[['canonical_title', 'review_text', 'review_text_clean']].head(5))

print('TMDb changed samples:')
display(tmdb_changed[['canonical_title', 'review_text', 'review_text_clean']].head(5))

## 6) Lineage Evidence Checks

In [ ]:
def lineage_completeness(df: pd.DataFrame) -> pd.DataFrame:
    required = ['bronze_file_name', 'bronze_source_type', 'processed_at']
    rows = []
    for col in required:
        rows.append({
            'column': col,
            'not_null_rows': int(df[col].notna().sum()),
            'total_rows': int(len(df)),
        })
    return pd.DataFrame(rows)

lineage_completeness(df_titles), lineage_completeness(df_rt_reviews), lineage_completeness(df_tmdb_reviews)

## 7) Compact Workshop 2 Summary

In [ ]:
summary = pd.DataFrame([
    {
        'dataset': 'silver_titles',
        'rows': len(df_titles),
        'cols': df_titles.shape[1],
        'unique_sources': df_titles['source'].nunique(dropna=True),
    },
    {
        'dataset': 'silver_rt_reviews',
        'rows': len(df_rt_reviews),
        'cols': df_rt_reviews.shape[1],
        'unique_sources': df_rt_reviews['source'].nunique(dropna=True),
    },
    {
        'dataset': 'silver_tmdb_reviews',
        'rows': len(df_tmdb_reviews),
        'cols': df_tmdb_reviews.shape[1],
        'unique_sources': df_tmdb_reviews['source'].nunique(dropna=True),
    },
])

summary